Mode	What you receive	When to use it
"messages"	LLM output chunks + metadata	Display the AI's answer as it's generated
"updates"	What each node returned	Know which node finished and what it changed
"values"	The complete graph state after each step	Inspect the state as the graph runs
"custom"	Data you manually send using a writer	Send progress messages such as "Searching..."
"checkpoints"	Checkpoint creation events	Monitor saved graph state
"tasks"	Task start/finish events and results	Monitor execution details
"debug"	Detailed execution information	Debug the graph

start node has values after finish then first node staart it has messages inside while executing once finished the updates happens + values then the next node start

In [2]:
import os

from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END
from pprint import pprint
from langgraph.checkpoint.memory import InMemorySaver
from uuid import uuid4

load_dotenv()


# 1. LLM
model = ChatOpenAI(
    model="meta/muse-glimmer-30b",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0.2,
    max_tokens=500,
    reasoning_effort="low"
)

checkpointer = InMemorySaver()


class State(BaseModel):
    question: str 
    research: str = ""
    answer: str = ""


def research_node(state: State):
    print("\n--- Research node executing ---")

    response = model.invoke(
        f"Give me 3 important facts about: {state.question}"
    )

    return {"research": response.content}


def answer_node(state: State):
    print("\n--- Answer node executing ---")

    response = model.invoke(
        f"Answer this question using the following research:\n"
        f"{state.research}\n"
        f"Question: {state.question}"
    )

    return {"answer": response.content}


builder = StateGraph(State)

builder.add_node("research", research_node)
builder.add_node("answer", answer_node)

builder.add_edge(START, "research")
builder.add_edge("research", "answer")
builder.add_edge("answer", END)

graph = builder.compile(checkpointer=checkpointer)

In [16]:


inputs = {
    "question": "What is Retrieval Augmented Generation?"
}

config = {
        "configurable": {
            "thread_id": str(uuid4())
        }
    }

modes = ["updates", "values", "messages", "custom"]

for mode, chunk in graph.stream(
    inputs,
    config=config,
    stream_mode=modes
):
    print(f"\n--- {mode.upper()} ---")

    if mode == "updates":
        pprint(chunk)

    elif mode == "values":
        pprint(chunk)

    elif mode == "custom":
        pprint(chunk)


--- VALUES ---
{'question': 'What is Retrieval Augmented Generation?'}

--- Research node executing ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES ---

--- MESSAGES

In [3]:

import asyncio


class State(BaseModel):
    question: str
    research: str = ""
    answer: str = ""


async def research_node(state: State):

    print("\n--- Research node executing ---")

    prompt = f"Give me 3 important facts about: {state.question}"

    result = ""

    async for chunk in model.astream(prompt):
        result += chunk.content

    return {"research": result}


async def answer_node(state: State):

    print("\n--- Answer node executing ---")

    prompt = f"""
    Answer this question using the following research:

    {state.research}

    Question: {state.question}
    """

    result = ""

    async for chunk in model.astream(prompt):
        result += chunk.content

    return {"answer": result}


builder = StateGraph(State)

builder.add_node("research", research_node)
builder.add_node("answer", answer_node)

builder.add_edge(START, "research")
builder.add_edge("research", "answer")
builder.add_edge("answer", END)

graph = builder.compile(checkpointer=checkpointer)

In [4]:
async def run():
    inputs = {
        "question": "What is Retrieval Augmented Generation?"
    }

    config = {
        "configurable": {
            "thread_id": str(uuid4())
        }
    }

    events = []

    async for event in graph.astream_events(
        inputs,
        config=config,
        version="v2"
    ):
        events.append(event)
    
        pprint(events)
    return events

res = await run()

[{'data': {'input': {'question': 'What is Retrieval Augmented Generation?'}},
  'event': 'on_chain_start',
  'metadata': {'ls_integration': 'langgraph',
               'thread_id': 'b0a1a3bd-7f38-42e4-bc89-01288d450205'},
  'name': 'LangGraph',
  'parent_ids': [],
  'run_id': '01a1ae11-74d4-7704-854f-ad390a3090d9',
  'tags': []}]
[{'data': {'input': {'question': 'What is Retrieval Augmented Generation?'}},
  'event': 'on_chain_start',
  'metadata': {'ls_integration': 'langgraph',
               'thread_id': 'b0a1a3bd-7f38-42e4-bc89-01288d450205'},
  'name': 'LangGraph',
  'parent_ids': [],
  'run_id': '01a1ae11-74d4-7704-854f-ad390a3090d9',
  'tags': []},
 {'data': {'input': State(question='What is Retrieval Augmented Generation?', research='', answer='')},
  'event': 'on_chain_start',
  'metadata': {'langgraph_checkpoint_ns': 'research:ec6f1a28-c970-fcee-8b2c-7cde768201e2',
               'langgraph_node': 'research',
               'langgraph_path': ('__pregel_pull', 'research'),
   

In [16]:

for event in res:
    print(event['metadata'].get("langgraph_node"))
 

None
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
research
None
answer
answer
answer
answer
answer
answer
answer
answer
answer
answer
answer
answer
answer
answer
answer
answer
answer
answer
answer
answer
answ